# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prakritibhandari07/FlyRank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #2 — "The Content Performance Curve" (p.7).** Health score by content age peaks at
61-90 days (33.1), falls to a decay cliff at 271-365 days (14.0), then partially recovers at
365+ days (25.1), described as "proof refresh works" elsewhere in the paper (Finding #8, p.14).

**My methodology question — where does the label (health score) actually come from?**
Health score is defined on p.5 as `impressions (30 pts) + position (30 pts) + CTR (20 pts) +
scroll depth (20 pts)`. That means the outcome being tracked across the age curve is built
directly from the same signals (impressions, position) whose lifecycle the finding describes —
so "health peaks at 61-90 days" is partly a restatement of "impressions and position peak at
61-90 days" by construction, not an independent confirmation. The paper is careful about this
in the ML appendix ("the target itself is partly constructed from some of these inputs... read
this as model behavior, not as a standalone optimization order," p.27) but that caveat isn't
repeated on the earlier age-curve page where the same composite metric is the star. I'd ask:
would this curve still show a clean peak-then-cliff-then-recovery shape if health score were
replaced with a single non-composite metric like raw impressions, held separately from
position and CTR? And for the 365+ recovery specifically — is "refreshed" a property that was
observed, or partly a property of which pages a client *chose* to refresh (a page a team
already believed was worth saving)? If refresh was a choice made by someone, part of the
365+ lift is the choosing, not the update itself, and the paper's own evidence-standard rules
(p.4) would call that out as a confound to disclose, the way it correctly does for the small
`365+ × 361+` cell on p.14 ("strong survivor bias").

---

**ML Appendix, "What Predicts Growth?" (p.29).** Logistic regression reports 71% holdout
accuracy separating growing from declining pages, with content age as "the strongest negative
signal."

**My methodology question — does the validation design actually carry that number?**
The methodology page (p.36) describes the split for every sklearn model as a plain
"80/20 split," with no mention of grouping by brand, even though the underlying sample spans
57 brands and (per p.2/p.25) 61.8K content pieces pooled from those brands. If rows from the
same brand can land on both sides of that 80/20 split, the model can partly memorize
brand-level quirks (a brand's typical publishing cadence, template, or niche) rather than learn
a signal that generalizes to a brand it hasn't seen — which is exactly the scenario the honest
split in Section 2 below is designed to catch. I'd ask FlyRank's data team the same question
I'm about to ask of my own model: does 71% survive a brand-held-out split, or is some of that
number the model recognizing which brand a row came from? Reporting the random-split number
next to a brand-grouped number (as the paper's own base-rate discipline elsewhere would
suggest) would settle it either way, and either answer is a fine, publishable result — the gap
itself would be informative.

In [1]:
# No dataset call needed for this section — these are close-reading questions about the
# published PDF (docs/flyrank-seo-research-march-2026.pdf), not a query against my data.
# Page references used above, for traceability:
findings_reviewed = {
    "Finding #2 — The Content Performance Curve": "p.7 (health-by-age curve), cross-refs p.5 (health score definition), p.14, p.27",
    "ML Appendix — What Predicts Growth?": "p.29 (logistic regression, 71% holdout accuracy), cross-ref p.36 (methodology: 80/20 split)",
}
for k, v in findings_reviewed.items():
    print(f"{k}\n  -> {v}\n")

Finding #2 — The Content Performance Curve
  -> p.7 (health-by-age curve), cross-refs p.5 (health score definition), p.14, p.27

ML Appendix — What Predicts Growth?
  -> p.29 (logistic regression, 71% holdout accuracy), cross-ref p.36 (methodology: 80/20 split)



## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Week-5 already trained the Random Forest under a `client_id`-grouped split (`GroupShuffleSplit`,
75/25). To make the honest-split improvement visible as a genuine **before/after**, I rebuilt
the *dishonest* comparison point here: the same features, same model, same hyperparameters,
but a naive **random row-level** 75/25 split that ignores that many rows share a `client_id`.
Both models are then scored on their own held-out test set with `precision@K` next to the test
set's base rate, per the leakage-hunting skill's rule that every score needs its base rate
printed alongside it.

In [5]:
!git clone https://github.com/Prakritibhandari07/FlyRank-ml-internship.git
%cd FlyRank-ml-internship

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Rows: {len(df)}, base rate: {df['is_declining_label'].mean():.3f}, clients: {df['client_id'].nunique()}")

def baseline_score(frame):
    stale = frame["freshness_tier"].isin(["91-180", "181+"]).astype(int)
    measurable = ((frame["impressions_90d"] >= 100) & (frame["sessions_90d"] > 0)).astype(int)
    tier_median_ctr = frame.groupby("position_tier", observed=True)["ctr"].transform("median")
    ctr_underperform = np.where(measurable == 1, (frame["ctr"] < 0.7 * tier_median_ctr).astype(int), 0)
    return stale * measurable * ctr_underperform * frame["impressions_90d"]

df["baseline_score"] = baseline_score(df)

# Same forbidden list and feature set as Week 5 — see Section 3 for the leakage re-audit.
FORBIDDEN = ["trend_direction", "trend_pct", "is_declining_label",
             "impressions_last_30d", "impressions_prev_30d",
             "content_id", "client_id", "baseline_score"]

numeric_features = [
    "days_since_last_update", "content_age_days", "impressions_90d", "clicks_90d",
    "pageviews_90d", "sessions_90d", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "search_volume", "competition", "cpc",
    "word_count", "char_count",
]
categorical_features = ["position_tier", "freshness_tier", "content_type", "main_intent",
                         "competition_level", "age_tier"]
nan_prone_cols = ["search_volume", "competition", "cpc", "word_count", "char_count",
                   "engagement_rate", "scroll_rate", "ai_traffic_pct"]

work = df.copy()
has_flag_cols = []
for col in nan_prone_cols:
    flag_col = f"has_{col}"
    work[flag_col] = work[col].notna().astype(int)
    work[col] = work[col].fillna(0)
    has_flag_cols.append(flag_col)
for col in categorical_features:
    work[col] = work[col].fillna("unknown")

feature_cols = numeric_features + has_flag_cols
X = pd.get_dummies(work[feature_cols + categorical_features], columns=categorical_features, drop_first=True)
y = work["is_declining_label"]
groups = df["client_id"]
assert X.isna().sum().sum() == 0
assert not any(c in X.columns for c in FORBIDDEN), "a forbidden column leaked into X"
print(f"Feature matrix: {X.shape}")

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

Ks = [20, 50, 100, 500]

def run_split(train_idx, test_idx, label):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1)
    rf.fit(X_train, y_train)
    scores = rf.predict_proba(X_test)[:, 1]
    row = {"split": label, "n_test": len(y_test), "base_rate": round(y_test.mean(), 3)}
    for k in Ks:
        row[f"precision@{k}"] = round(precision_at_k(scores, y_test.values, k), 3)
    tr_clients = set(groups.iloc[train_idx])
    te_clients = set(groups.iloc[test_idx])
    row["client_overlap_train_test"] = len(tr_clients & te_clients)
    return row, rf, X_test, y_test, scores

# BEFORE — naive random row-level split (dishonest: ignores repeating client_id)
tr_idx_r, te_idx_r = train_test_split(np.arange(len(df)), test_size=0.25,
                                       random_state=RANDOM_SEED, stratify=y)
row_random, rf_random, *_ = run_split(tr_idx_r, te_idx_r, "BEFORE: random split (naive)")

# AFTER — grouped by client_id (honest; this is what Week 5 actually used)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
tr_idx_g, te_idx_g = next(gss.split(df, groups=groups))
row_grouped, rf_grouped, X_test_g, y_test_g, test_scores_g = run_split(
    tr_idx_g, te_idx_g, "AFTER: grouped by client_id (honest)")

comparison = pd.DataFrame([row_random, row_grouped])
comparison

Cloning into 'FlyRank-ml-internship'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 164 (delta 69), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (164/164), 1.88 MiB | 9.78 MiB/s, done.
Resolving deltas: 100% (69/69), done.
/content/FlyRank-ml-internship
Rows: 30000, base rate: 0.542, clients: 32
Feature matrix: (30000, 43)


,split,n_test,base_rate,precision@20,precision@50,precision@100,precision@500,client_overlap_train_test
0,BEFORE: random split (naive),7500,0.542,0.9,0.88,0.89,0.866,31
1,AFTER: grouped by client_id (honest),7115,0.517,0.5,0.54,0.57,0.584,0


**Reading the before/after table:** the random split leaves 31 of 32 clients on both
sides of train and test (`client_overlap_train_test`), so precision@20 looks like 0.90 — the
model is largely recognizing clients it already memorized, not ranking unseen content. The
client-grouped split (0 client overlap by construction — this is what Week 5 used) drops
precision@20 to 0.50, next to a test-set base rate of 0.52. That ~40-point gap at K=20 is
itself the finding: **most of the "before" score was memorization, not generalization.** The
honest number (0.50–0.58 across K, versus a 0.52 base rate) is modest — a small measured edge,
not a strong one — and that's the number that should be reported and trusted going forward.

## 3. Leakage audit
*The same hunt from Week 3, on your final feature set.*

Re-running the attack checklist from `hunting-leakage-and-validating` against the exact
feature set trained above (the honest, client-grouped model).


In [6]:
print("--- Leakage audit checklist ---\n")

# 1) No label-derived or forbidden columns actually made it into the feature matrix.
leaked_cols = [c for c in FORBIDDEN if c in X.columns]
print(f"1) Forbidden columns present in X: {leaked_cols}  (expect: [])")

# 2) Population selection check: did I filter rows using anything from the outcome window?
print(f"2) Population selection: all {len(df)} rows used, no filter on trend_direction, "
      f"trend_pct, or any impressions_last/prev_30d column before the split. "
      f"(avg_position == 0 for 1,205 rows means 'no data', per the data dictionary — those rows "
      f"are kept, not dropped, so there's no outcome-window-based survivorship here.)")

# 3) Top feature importances on the honest model — sanity-check nothing looks "too good".
importances = pd.Series(rf_grouped.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\n3) Top 8 feature importances, honest grouped model:")
print(importances.head(8).round(4))
print("   No single feature towers over the rest (impressions_90d leads at ~17%) — "
      "no near-perfect single predictor, unlike the injected case below.")

# 4) The actual attack: deliberately inject a known label-derived feature and watch the score
#    jump toward 1.0. is_declining_label is computed FROM trend_direction, which is computed
#    FROM trend_pct (per skills/flyrank/flyrank-data "the label trap") — trend_pct is exactly
#    the kind of sibling-of-the-label column the leakage skill warns about.
X_leaky = X.copy()
X_leaky["LEAKY_trend_pct"] = df["trend_pct"].values
X_train_leaky, X_test_leaky = X_leaky.iloc[tr_idx_g], X_leaky.iloc[te_idx_g]
rf_leaky = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1)
rf_leaky.fit(X_train_leaky, y.iloc[tr_idx_g])
leaky_scores = rf_leaky.predict_proba(X_test_leaky)[:, 1]
leaky_p50 = precision_at_k(leaky_scores, y.iloc[te_idx_g].values, 50)
honest_p50 = row_grouped["precision@50"]
print(f"\n4) Deliberate leak test — inject trend_pct (sibling of the label) as a feature:")
print(f"   precision@50 WITHOUT it (honest): {honest_p50}")
print(f"   precision@50 WITH it (leaky):     {leaky_p50:.3f}")
leaky_importance = pd.Series(rf_leaky.feature_importances_, index=X_leaky.columns).sort_values(ascending=False)
print(f"   LEAKY_trend_pct grabs {leaky_importance.iloc[0]*100:.1f}% of feature importance — "
      f"the exact 'one feature towers over all others, score near-perfect' confession the "
      f"skill describes. Confirms the test harness would catch real leakage, and confirms "
      f"trend_pct/trend_direction correctly stay OUT of the real feature set above.")

# 5) Error examples — where does the honest model get it wrong?
test_df_g = df.iloc[te_idx_g].copy()
test_df_g["model_score"] = test_scores_g
test_df_g["true_label"] = y_test_g.values
top50 = test_df_g.sort_values("model_score", ascending=False).head(50)
false_positives = top50[top50["true_label"] == 0]
print(f"\n5) Of the top-50 pages the honest model flags as highest decline-risk, "
      f"{len(false_positives)} are false positives (flagged 'declining', actually not).")
print("   False positives by position_tier:")
print("  ", dict(false_positives["position_tier"].value_counts()))
print("\n   Three example false positives:")
cols = ["content_id", "position_tier", "avg_position", "impressions_90d",
        "content_age_days", "trend_direction", "model_score"]
print(false_positives[cols].head(3).to_string(index=False))

--- Leakage audit checklist ---

1) Forbidden columns present in X: []  (expect: [])
2) Population selection: all 30000 rows used, no filter on trend_direction, trend_pct, or any impressions_last/prev_30d column before the split. (avg_position == 0 for 1,205 rows means 'no data', per the data dictionary — those rows are kept, not dropped, so there's no outcome-window-based survivorship here.)

3) Top 8 feature importances, honest grouped model:
impressions_90d        0.1742
avg_position           0.1449
content_age_days       0.1073
char_count             0.0528
word_count             0.0522
position_tier_top_3    0.0397
scroll_rate            0.0343
clicks_90d             0.0340
dtype: float64
   No single feature towers over the rest (impressions_90d leads at ~17%) — no near-perfect single predictor, unlike the injected case below.

4) Deliberate leak test — inject trend_pct (sibling of the label) as a feature:
   precision@50 WITHOUT it (honest): 0.54
   precision@50 WITH it (leaky)

**What the false positives tell me:** all three examples sit at `page_3_5` or `striking`
position tiers with moderate age and moderate visibility, and the model scored them ~0.82 as
decline-risk — but their real `trend_direction` was `up` or `stable`. The model appears to be
leaning on "older, lower-position content with moderate impressions" as a decline signal, which
is directionally sensible but not deterministic: a page can sit exactly in that profile and
still be climbing. 11 of the 23 false positives in the top-50 cluster in `page_3_5`, which
matches a known baseline weak spot from Week 4 — this is a population the model (and the rule
before it) both over-flag, and it's the honest reason `precision@50` sits at 0.54, not near
1.0. That's a useful, disclosable limitation, not a bug to hide.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured,
directional, decision-support.*

**My boldest sentence, as I might casually have written it in Week 5 or in `model_report.md`:**
> "Our model predicts which content will decline, so the team should trust its flags to decide
> what to refresh."

That sentence claims prediction-as-fact and recommends automatic trust — neither is supported
by what the honest evaluation actually shows.

**Rewritten in safe language, using the numbers from Section 2/3 above:**
> On a client-held-out split (0 client overlap between train and test), the Random Forest
> ranks pages by decline risk with `precision@50` of 0.54 against a test-set base rate of
> 0.52 — a small, measured edge over chance, and roughly comparable to the rule-based
> baseline (`precision@50` = 0.54) rather than a decisive improvement over it. The model is a
> **decision-support ranking** for prioritizing manual review, not a prediction of what will
> happen to any individual page — the false-positive review above shows it can and does
> mis-flag legitimately growing or stable content, especially in the `page_3_5` position tier.
> Its "random-split" number (`precision@20` = 0.90) should not be quoted on its own: that
> figure reflects client memorization, not out-of-sample skill, and is retired in favor of the
> honest grouped number throughout this notebook and the queue in `outputs/model_report.md`.

In [7]:
# The numbers the rewritten claim above cites, pulled directly from the honest run —
# keeping the receipts next to the rewrite rather than restating them from memory.
print("Honest (client-grouped) precision@K vs baseline, same test set:\n")
for k in Ks:
    b = precision_at_k(test_df_g["baseline_score"].values, y_test_g.values, k)
    m = precision_at_k(test_scores_g, y_test_g.values, k)
    print(f"  K={k:>3}: baseline={b:.3f}  model={m:.3f}  base_rate={y_test_g.mean():.3f}")

print(f"\nFor reference — the retired, not-to-be-quoted number: "
      f"random-split precision@20 = {row_random['precision@20']} "
      f"(client_overlap_train_test = {row_random['client_overlap_train_test']} of "
      f"{df['client_id'].nunique()} clients).")

Honest (client-grouped) precision@K vs baseline, same test set:

  K= 20: baseline=0.450  model=0.500  base_rate=0.517
  K= 50: baseline=0.540  model=0.540  base_rate=0.517
  K=100: baseline=0.510  model=0.570  base_rate=0.517
  K=500: baseline=0.544  model=0.584  base_rate=0.517

For reference — the retired, not-to-be-quoted number: random-split precision@20 = 0.9 (client_overlap_train_test = 31 of 32 clients).


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.